# 5.7 — Bag-of-Words, n-grame, TF-IDF și matrici rare

Material ONIA · Aplicații în Python și scikit-learn. Datele sunt sintetice. Pentru a rula: `pip install scikit-learn numpy scipy matplotlib`.

## Construirea vocabularului și a matricii BoW de la zero

In [ ]:
texte = ["tenis sport", "tenis sport date", "algoritm"]
tokeni = [text.lower().split() for text in texte]
vocabular = sorted({cuvant for doc in tokeni for cuvant in doc})

matrice = []
for doc in tokeni:
    linie = []
    for cuvant in vocabular:
        linie.append(doc.count(cuvant))
    matrice.append(linie)

print("Vocabular:", vocabular)
for i, linie in enumerate(matrice, 1):
    print("D", i, linie)


## Același lucru cu CountVectorizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

texte = ["tenis sport", "tenis sport date", "algoritm"]
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texte)
print(vectorizer.get_feature_names_out().tolist())
print(X.toarray())  # doar pentru un corpus minuscul


## Compararea numărării cu prezența binară

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

texte = ["ofertă ofertă ofertă", "ofertă verificată", "mesaj simplu"]
for mod in [False, True]:
    v = CountVectorizer(binary=mod)
    X = v.fit_transform(texte)
    print("binary=", mod, "vocabular=", v.get_feature_names_out().tolist())
    print(X.toarray())


## N-grame calculate manual

In [ ]:
tokeni = ["nu", "este", "foarte", "bun"]
for n in [1, 2, 3]:
    ngrame = []
    for i in range(len(tokeni) - n + 1):
        ngrame.append(" ".join(tokeni[i:i+n]))
    print(f"{n}-grame:", ngrame)


## Compararea unigrame–bigrame în scikit-learn

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

texte = ["nu recomand produsul", "recomand produsul", "produs foarte bun"]
for limita in [(1, 1), (1, 2)]:
    v = CountVectorizer(ngram_range=limita)
    X = v.fit_transform(texte)
    print(limita, "->", len(v.get_feature_names_out()), "caracteristici")
    print(v.get_feature_names_out().tolist())


## N-grame de caractere pentru greșeli de tastare

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

texte = ["programare", "progrmare", "fotbal"]
v = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
X = v.fit_transform(texte)
S = cosine_similarity(X)
print("Similaritate programare/progrmare:", round(S[0, 1], 3))
print("Similaritate programare/fotbal:", round(S[0, 2], 3))


## Păstrăm sau nu tokenii de o literă?

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

texte = ["C și R sunt limbaje", "X este o variabilă"]
implicit = CountVectorizer()
extins = CountVectorizer(token_pattern=r"(?u)\b\w+\b")
print("Implicit:", implicit.fit(texte).get_feature_names_out().tolist())
print("Cu o literă:", extins.fit(texte).get_feature_names_out().tolist())


## TF și DF calculate de la zero

In [ ]:
from collections import Counter

texte = ["python python date", "python model", "date model", "python cod"]
tokeni = [text.split() for text in texte]
tf_d1 = Counter(tokeni[0])
df = Counter()
for doc in tokeni:
    for termen in set(doc):
        df[termen] += 1
print("TF în D1:", tf_d1)
print("DF în corpus:", df)


## Calculul complet TF-IDF pentru un document

In [ ]:
from math import log, sqrt

documente = ["python python date", "python cod", "date cod"]
tokeni = [text.split() for text in documente]
vocabular = ["python", "date", "cod"]
N = len(tokeni)

vector = []
for termen in vocabular:
    tf = tokeni[0].count(termen)
    df = sum(termen in doc for doc in tokeni)
    idf = log((1 + N) / (1 + df)) + 1
    vector.append(tf * idf)
    print(termen, "TF=", tf, "DF=", df, "IDF=", round(idf, 4))

norma = sqrt(sum(x*x for x in vector))
normalizat = [x/norma for x in vector] if norma else vector
print("TF-IDF brut:", [round(x, 4) for x in vector])
print("TF-IDF L2:", [round(x, 4) for x in normalizat])


## Verificare: același vector cu TfidfVectorizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

documente = ["python python date", "python cod", "date cod"]
v = TfidfVectorizer(smooth_idf=True, norm="l2", sublinear_tf=False)
X = v.fit_transform(documente)
print(v.get_feature_names_out().tolist())
print(X.toarray()[0].round(4))


## Efectul normalizării la documente de lungimi diferite

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

texte = ["python date", "python date python date", "fotbal sport"]
for norm in [None, "l2"]:
    v = TfidfVectorizer(norm=norm)
    X = v.fit_transform(texte)
    print("norm=", norm)
    print(X.toarray().round(3))


## Comparație directă: frecvența brută versus logaritmică

In [ ]:
from math import log

for tf in [1, 2, 5, 10, 100]:
    liniar = tf
    subliniar = 1 + log(tf)
    print(f"TF={tf:3d}: brut={liniar:6.2f}, sublinear={subliniar:6.2f}")


## Selecție supervizată după vectorizare, fără leakage

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10000)),
    ("selectie", SelectKBest(chi2, k=1000)),
    ("clasificator", LogisticRegression(max_iter=1000))
])
# model.fit(X_train, y_train)
# Folosește corpus real, cu cel puțin 1000 de caracteristici distincte.


## Inspectarea reprezentării CSR

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

texte = ["python cod", "python date", "sport fotbal"]
X = CountVectorizer().fit_transform(texte)
print("Tip:", type(X).__name__)
print("Dimensiune:", X.shape, "valori nenule:", X.nnz)
print("data:", X.data.tolist())
print("indices:", X.indices.tolist())
print("indptr:", X.indptr.tolist())
print("Matrice (numai pentru demo):")
print(X.toarray())


## Separarea corectă train / test

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

train = ["python algoritmi", "matematică date", "python modele"]
test = ["algoritmi noi", "robotică și python"]
v = TfidfVectorizer()
X_train = v.fit_transform(train)
X_test = v.transform(test)   # NU fit_transform(test)

print("Aceleași coloane:", X_train.shape[1] == X_test.shape[1])
print("Vocabular:", v.get_feature_names_out().tolist())


## HashingVectorizer pe texte noi, fără fit

In [ ]:
from sklearn.feature_extraction.text import HashingVectorizer

v = HashingVectorizer(n_features=16, alternate_sign=False, norm=None)
X1 = v.transform(["python date", "sport meci"])
X2 = v.transform(["python model nou"])
print("Număr coloane fix:", X1.shape, X2.shape)
print("Valori nenule:", X1.nnz, X2.nnz)


## TF-IDF → SVD → vectori de dimensiune redusă

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

texte = [
    "tenis sport meci", "fotbal stadion sport", "echipă sport meci",
    "python cod date", "algoritmi python date", "model cod date"
]
X = TfidfVectorizer().fit_transform(texte)
Z = TruncatedSVD(n_components=2, random_state=42).fit_transform(X)
print("Înainte:", X.shape, "după:", Z.shape)
print(Z.round(3))


## FeatureUnion: TF-IDF de cuvinte + TF-IDF de caractere

In [ ]:
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

reprezentare = FeatureUnion([
    ("cuvinte", TfidfVectorizer(ngram_range=(1, 2))),
    ("caractere", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)))
])
model = Pipeline([
    ("vectorizare", reprezentare),
    ("clasificare", LogisticRegression(max_iter=1000))
])
X_train = ["ofertă urgentă", "reducere mare", "proiect python", "lecție matematică"]
y_train = ["spam", "spam", "educație", "educație"]
model.fit(X_train, y_train)
print(model.predict(["oferata urgenta", "proiect algoritmi"]))


## Pipeline corect pentru clasificarea textului și validare

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

texte = [
    "tenis turneu", "fotbal meci", "echipă stadion", "sport competiție",
    "cod python", "date algoritm", "programare python", "software aplicație"
]
etichete = ["sport"] * 4 + ["tech"] * 4
model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clf", LinearSVC(C=1.0))
])
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
print(cross_val_score(model, texte, etichete, cv=cv, scoring="f1_macro"))


## Motor de căutare lexicală cu TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documente = [
    "elevii învață python și algoritmi",
    "echipa a câștigat meciul de fotbal",
    "date și modele de inteligență artificială",
    "profesorul explică exerciții python"
]
v = TfidfVectorizer(ngram_range=(1, 2))
X = v.fit_transform(documente)
interogare = "exerciții python"
q = v.transform([interogare])
scoruri = cosine_similarity(q, X).ravel()
ordine = scoruri.argsort()[::-1]
for i in ordine:
    print(f"{scoruri[i]:.3f} | {documente[i]}")


## Exerciții pentru experimentare

1. Schimbă corpusul astfel încât un termen să apară în toate documentele; verifică IDF.
2. Compară `ngram_range=(1,1)` cu `(1,2)` la texte cu negații.
3. Creează un set de date cu greșeli de tastare și compară vectorizatorul pe cuvinte cu cel pe caractere.
4. Evită `fit` pe test în toate experimentele.
5. Pentru un corpus real, investighează costul memoriei și valoarea `nnz`.